# Fine-Tuning GPT for Multilingual Causal QA – FinCausal 2026

> Fine-tunes `gpt-4.1-mini` on English and Spanish extractive QA data for the FinCausal 2026 shared task.  
> Covers data loading, chat format conversion, JSONL export, token counting, and job submission via the OpenAI API.

## 1. Import Libraries

In [ ]:
# ============================================
# Library Imports
# ============================================

import os
import json
import pandas as pd
from datasets import load_dataset
import tiktoken
from openai import OpenAI

## 2. Load Datasets

### 2.1 English Dataset


In [ ]:
train_df_en = pd.read_csv("Dataset/EN/train_80_en.csv",sep=";")
dev_df_en = pd.read_csv("Dataset/EN/dev_20_en.csv",sep=";")

### 2.2 Spanish Dataset

In [ ]:
train_df_es = pd.read_csv("Dataset/ES/train_80_es.csv",sep=";")
dev_df_es = pd.read_csv("Dataset/ES/dev_20_es.csv",sep=";")

## 3. Multilingual Dataset Preparation

In [ ]:
# Concatenate and shuffle English + Spanish training sets
train_df = pd.concat([train_df_en, train_df_es], ignore_index=True)
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Concatenate and shuffle English + Spanish development sets
dev_df = pd.concat([dev_df_en, dev_df_es], ignore_index=True)
dev_df = dev_df.sample(frac=1, random_state=42).reset_index(drop=True)

## 4. Dataset Statistics

In [ ]:
def print_dataset_statistics(train_df, dev_df):
    print("Train Dataset Statistics")
    print("-------------------------")
    print(f"Number of samples: {len(train_df)}")
    print(f"Average context length: {train_df['context'].str.len().mean():.2f}")
    print(f"Average question length: {train_df['question'].str.len().mean():.2f}")
    print(f"Average answer length: {train_df['answer'].str.len().mean():.2f}")
    print()
    print("Development Dataset Statistics")
    print("-------------------------------")
    print(f"Number of samples: {len(dev_df)}")
    print(f"Average context length: {dev_df['context'].str.len().mean():.2f}")
    print(f"Average question length: {dev_df['question'].str.len().mean():.2f}")
    print(f"Average answer length: {dev_df['answer'].str.len().mean():.2f}")

print_dataset_statistics(train_df, dev_df)

## 5. Convert to Chat Format for Fine-Tuning

In [ ]:
SYSTEM_MESSAGE = """You are a causal analysis assistant. You will be given a passage and a question about financial causal relationships. Your task is to answer the question by extracting the relevant information verbatim from the passage. Do not paraphrase or infer beyond what is explicitly stated in the text."""

USER_FORMAT = "Context: {context}\n\nQuestion: {question}"

def convert_to_chat_format(row):
    user_message = USER_FORMAT.format(context=row['context'], question=row['question'])
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_MESSAGE},
            {'role': 'user', 'content': user_message},
            {'role': 'assistant', 'content': row['answer']}
        ]
    }

train_data = train_df.apply(convert_to_chat_format, axis=1).tolist()
dev_data = dev_df.apply(convert_to_chat_format, axis=1).tolist()

## 6. Save and Verify JSONL Files

In [ ]:
def save_jsonl(data, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"Saved {filename} ({len(data)} samples)")

def load_jsonl(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

print("\nSaving files...")
save_jsonl(train_data, 'train.jsonl')
save_jsonl(dev_data, 'eval.jsonl')

print("\nVerification:")
loaded_train = load_jsonl('train.jsonl')
loaded_eval = load_jsonl('eval.jsonl')
print(f"train.jsonl has {len(loaded_train)} samples")
print(f"eval.jsonl has {len(loaded_eval)} samples")

### Summary: Files Ready for Fine-Tuning

In [ ]:
print("\n" + "="*70)
print("✅ DONE!")
print("="*70)
print(f"Files created:")
print(f"  - train.jsonl ({len(train_data)} samples)")
print(f"  - eval.jsonl ({len(dev_data)} samples)")
print(f"  - data_stats.json (statistics)")
print("\nReady for fine-tuning!")

## 7. Load JSONL Files as Hugging Face Datasets

In [ ]:
train_dataset = load_dataset('json', data_files='train.jsonl', split='train')
eval_dataset = load_dataset('json', data_files='eval.jsonl', split='train')
print(train_dataset[0])

### Verify Dataset Sizes

In [ ]:
print("Train size:", len(train_dataset))
print("Eval size:", len(eval_dataset))

## 8. Token Counting for JSONL Files

In [ ]:
enc = tiktoken.get_encoding("o200k_base")

def count_tokens_jsonl(path, text_keys=("text",)):
    total = 0
    n = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            text = None
            for k in text_keys:
                if k in obj and isinstance(obj[k], str):
                    text = obj[k]
                    break
            if text is None:
                text = json.dumps(obj, ensure_ascii=False)
            total += len(enc.encode(text))
            n += 1
    return n, total

train_n, train_tokens = count_tokens_jsonl("train.jsonl", text_keys=("text", "prompt", "input"))
eval_n, eval_tokens = count_tokens_jsonl("eval.jsonl", text_keys=("text", "prompt", "input"))

print("Train rows:", train_n, "Train tokens:", train_tokens)
print("Eval rows:", eval_n, "Eval tokens:", eval_tokens)
print("Total tokens:", train_tokens + eval_tokens)

## 9. Fine-Tuning

### 9.1 Initialise OpenAI Client

In [ ]:
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set. Add it to your environment variables.")
client = OpenAI(api_key=api_key)

### 9.2 Upload Files and Create Fine-Tuning Job

In [ ]:
train_file = client.files.create(file=open("train.jsonl", "rb"), purpose="fine-tune")
eval_file = client.files.create(file=open("eval.jsonl", "rb"), purpose="fine-tune")

job = client.fine_tuning.jobs.create(
    model="gpt-4.1-mini-2025-04-14",
    training_file=train_file.id,
    validation_file=eval_file.id,
    method={
        "type": "supervised",
        "supervised": {
            "hyperparameters": {
                "n_epochs": 3,
                "batch_size": 4,
                "learning_rate_multiplier": 0.1
            }
        }
    }
)

print("Fine-tuning job created:", job.id)

### 9.3 Upload Training File (Direct)

In [ ]:
client.files.create(
  file=open("train.jsonl", "rb"),
  purpose="fine-tune"
)

### 9.4 Submit Fine-Tuning Job (Direct)

In [ ]:
client.fine_tuning.jobs.create(
  training_file='file-LWejM3AeP9rhNiUT83xAkT',
  model="gpt-4.1-2025-04-14"
)

## 10. Fine-Tuning Job Output and Monitoring

### Fine-Tuning Job Response Example

When a fine-tuning job is created, the API returns an object similar to:

FineTuningJob  
- **id**: `ftjob-xxxxxxxxxxxxxxxx`  
- **status**: `validating_files`  
- **model**: `gpt-4.1-2025-04-14`  
- **training_file**: `file-xxxxxxxx`  
- **fine_tuned_model**: `None`  

### Explanation

- **id** → The job identifier used to monitor progress.  
- **status** → Indicates the current stage (e.g., validating_files, running, succeeded, failed).  
- **fine_tuned_model** → Becomes available only after the job completes successfully.

### Monitor Job Progress

You can track the status of this fine-tuning job directly in the OpenAI dashboard:

https://platform.openai.com/finetune/ftjob-nqi5tdiTKcD3acPReEFa2vdH?filter=all

---

### Reference: Fine-Tuning Documentation

Official guide: https://developers.openai.com/api/docs/guides/supervised-fine-tuning/